# Preprocessing: Before & After Illumination Correction

Demonstrates the illumination correction and background subtraction pipeline on actual TMEM106B imaging data across **all timepoints**.

The IC field is the **pixelwise median** across a seeded random sample of each timepoint's images (median is robust to bright-cell outliers; seeding makes it reproducible), Gaussian-smoothed, then normalized by its **mean** so the field is centered on 1 — it both attenuates the bright center and amplifies dim/vignetted corners (values can be `< 1`). The correction model is `(raw − darkfield) / flatfield`; this notebook demonstrates the flatfield path only (`darkfield=None`).

**Key functions:**
- `calculate_ic_field()` — pixelwise-median IC field from a set of images (per-well or per-plate), seeded sampling, threaded I/O; `estimate_darkfield=True` also returns a scalar darkfield
- `calculate_ic_fields_by_timepoint()` — compute IC fields for all timepoints in one call
- `preprocess_with_lookup()` — auto-selects IC field by timepoint and applies preprocessing (optional `darkfield=`)
- `apply_ic_field()` — subtract darkfield (if given), then divide by the IC field
- `subtract_background()` — rolling ball background subtraction

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tmem_align.io import read_image
from tmem_align.preprocess import (
    calculate_ic_field,
    calculate_ic_fields_by_timepoint,
    apply_ic_field,
    subtract_background,
    preprocess_image,
    preprocess_with_lookup,
)

## 1. Load Data

Discover all timepoints and load a test image from one timepoint.

In [ ]:
from tmem_align.nd2_tools import inspect_nd2

PLATE_DIR = Path("/Users/pmihack/claire/tmem_2026/data/260213_Feb16recopy_HYdiff_landingpadlines_survival_384well1")
timepoint_dirs = sorted(d for d in PLATE_DIR.iterdir() if d.is_dir())
print(f"Timepoints: {len(timepoint_dirs)}")
for d in timepoint_dirs:
    print(f"  {d.name}: {len(list(d.glob('*.nd2')))} images")

# Load a test image from one timepoint
test_tp = timepoint_dirs[4]  # 20260228
test_files = sorted(test_tp.glob("*.nd2"))
test_file = [f for f in test_files if "WellE05" in f.name][0]
test_img = read_image(test_file)

# Channel names in the ND2 are just excitation wavelengths (405/561/488nm), not dyes.
# Biologist confirmed the actual labels: only mCherry (561) and mNeonGreen (488) were used.
# No Hoechst/DAPI — the 405 channel is autofluorescence/background in the DAPI emission band.
DYE_BY_WAVELENGTH = {"561": "mCherry", "488": "mNeonGreen", "405": "autofluor/bg"}
raw_names = inspect_nd2(test_file)["channel_names"]
CHANNEL_NAMES = [
    f"{n} ({DYE_BY_WAVELENGTH[wl]})" if (wl := next((w for w in DYE_BY_WAVELENGTH if w in n), None)) else n
    for n in raw_names
]
print(f"\nTest image: {test_file.name}")
print(f"Shape: {test_img.shape}, dtype: {test_img.dtype}")
print(f"Channels: {CHANNEL_NAMES}")

## 2. Calculate IC Fields for All Timepoints

One IC field per channel per timepoint, computed in one call. Uses threaded I/O and 25% sampling.

In [ ]:
%%time
ic_fields = calculate_ic_fields_by_timepoint(PLATE_DIR, sample_fraction=0.25)
print(f"\nComputed IC fields for {len(ic_fields)} timepoints")
for tp_name, ic in ic_fields.items():
    print(f"  {tp_name}: shape={ic.shape}, range=[{ic.min():.2f}, {ic.max():.2f}]")

## 3. Visualize IC Fields — Single Timepoint

IC field for the test timepoint, normalized to a mean of 1. Values `> 1` mark the bright center-of-FOV (attenuated on division); values `< 1` mark dim/vignetted corners (amplified). Dividing by this field flattens the illumination profile.

In [ ]:
ic_field = ic_fields[test_tp.name]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, name) in enumerate(zip(axes, CHANNEL_NAMES)):
    im = ax.imshow(ic_field[i], cmap="inferno")
    ax.set_title(f"IC Field — {name}")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"IC Fields for {test_tp.name}", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Before / After: Illumination Correction

Side-by-side comparison of raw vs IC-corrected images for each channel.

In [ ]:
# Use preprocess_with_lookup — auto-selects IC field by timepoint
corrected = preprocess_with_lookup(test_file, ic_fields)

fig, axes = plt.subplots(3, 2, figsize=(12, 16))
for i, name in enumerate(CHANNEL_NAMES):
    raw_ch = test_img[i]
    cor_ch = corrected[i]
    vmin, vmax = np.percentile(raw_ch, [1, 99])

    axes[i, 0].imshow(raw_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 0].set_title(f"Raw — {name}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cor_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 1].set_title(f"IC Corrected — {name}")
    axes[i, 1].axis("off")

fig.suptitle("Before / After Illumination Correction (Well E05, Day 20)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Intensity Profiles

Line profiles across the image center showing how IC flattens the illumination gradient.

In [ ]:
from scipy.ndimage import uniform_filter1d

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mid_row = test_img.shape[1] // 2

for i, (ax, name) in enumerate(zip(axes, CHANNEL_NAMES)):
    raw_profile = test_img[i, mid_row, :].astype(float)
    cor_profile = corrected[i, mid_row, :].astype(float)

    raw_smooth = uniform_filter1d(raw_profile, 50)
    cor_smooth = uniform_filter1d(cor_profile, 50)

    ax.plot(raw_smooth, label="Raw", alpha=0.8)
    ax.plot(cor_smooth, label="Corrected", alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel("X position (px)")
    ax.set_ylabel("Intensity")
    ax.legend()

fig.suptitle("Horizontal Intensity Profiles (center row, smoothed)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Background Subtraction

Rolling ball background subtraction removes slowly varying background (autofluorescence, optical artifacts).

In [ ]:
# Full pipeline via preprocess_with_lookup: IC + background subtraction
fully_preprocessed = preprocess_with_lookup(test_file, ic_fields, background_radius=100)

fig, axes = plt.subplots(3, 3, figsize=(16, 16))
for i, name in enumerate(CHANNEL_NAMES):
    raw_ch = test_img[i]
    ic_ch = corrected[i]
    full_ch = fully_preprocessed[i]
    vmin, vmax = np.percentile(raw_ch, [1, 99])

    axes[i, 0].imshow(raw_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 0].set_title(f"Raw — {name}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(ic_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 1].set_title(f"IC Only — {name}")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(full_ch, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i, 2].set_title(f"IC + BG Sub — {name}")
    axes[i, 2].axis("off")

fig.suptitle("Full Preprocessing Pipeline Comparison", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
stats = []
for i, name in enumerate(CHANNEL_NAMES):
    for label, arr in [("Raw", test_img), ("IC Corrected", corrected), ("IC+BG Sub", fully_preprocessed)]:
        ch = arr[i].astype(float)
        stats.append({
            "Channel": name,
            "Stage": label,
            "Mean": f"{ch.mean():.1f}",
            "Std": f"{ch.std():.1f}",
            "CV%": f"{100 * ch.std() / ch.mean():.1f}" if ch.mean() > 0 else "N/A",
            "Min": int(ch.min()),
            "Max": int(ch.max()),
        })

pd.DataFrame(stats)

## 8. Cross-Timepoint IC Field Comparison

Are the IC fields stable across imaging sessions, or does the illumination drift? If they drift significantly, per-timepoint IC is essential.

In [ ]:
tp_names = list(ic_fields.keys())
n_tp = len(tp_names)

# Show IC field center-row profiles across timepoints for each channel
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ch_idx, (ax, ch_name) in enumerate(zip(axes, CHANNEL_NAMES)):
    for tp_name in tp_names:
        ic = ic_fields[tp_name]
        mid = ic.shape[-2] // 2
        profile = ic[ch_idx, mid, :] if ic.ndim == 3 else ic[mid, :]
        # Use short date label
        label = tp_name[:8]  # YYYYMMDD
        ax.plot(profile, alpha=0.6, label=label)
    ax.axhline(1.0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax.set_title(ch_name)
    ax.set_xlabel("X position (px)")
    ax.set_ylabel("IC field value (divisor, mean-centered on 1)")
    ax.legend(fontsize=7, ncol=2)

fig.suptitle("IC Field Center-Row Profiles Across All Timepoints", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify IC field drift: pairwise correlation between timepoints per channel
drift_data = []
for ch_idx, ch_name in enumerate(CHANNEL_NAMES):
    fields = []
    for tp_name in tp_names:
        ic = ic_fields[tp_name]
        f = ic[ch_idx].ravel() if ic.ndim == 3 else ic.ravel()
        fields.append(f)
    # Compare each timepoint to the first
    ref = fields[0]
    for i, (f, tp) in enumerate(zip(fields, tp_names)):
        corr = np.corrcoef(ref, f)[0, 1]
        max_diff = np.abs(ic_fields[tp_names[0]][ch_idx] - ic_fields[tp][ch_idx]).max() if ic_fields[tp].ndim == 3 else 0
        drift_data.append({
            "Channel": ch_name,
            "Timepoint": tp[:8],
            "Corr vs first": f"{corr:.4f}",
            "Max abs diff": f"{max_diff:.3f}",
        })

drift_df = pd.DataFrame(drift_data)
print("IC Field Stability (correlation and max absolute difference vs first timepoint):")
drift_df